In [57]:
import torch
import torch.nn as nn

In [58]:
import torchvision
from torchvision import datasets, transforms

In [59]:
from torch.utils.data import DataLoader

In [60]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [61]:
trainset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",              
    target_types="binary-category", # Returns 0 for cat, 1 for dog
    transform=transform,
    download=False
)
testset = datasets.OxfordIIITPet(
    root="./data",
    split="test",              
    target_types="binary-category", # Returns 0 for cat, 1 for dog
    transform=transform,
    download=False
)




In [62]:
trainset

Dataset OxfordIIITPet
    Number of datapoints: 3680
    Root location: ./data
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [63]:
testset

Dataset OxfordIIITPet
    Number of datapoints: 3669
    Root location: ./data
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [64]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64)

In [65]:
train_loader

In [66]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Input: 3 x 128 x 128
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # 32 x 64 x 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # 64 x 32 x 32
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            # 128 x 16 x 16 = 32768
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),

            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)

        x = x.view(x.size(0), -1)

        x = self.fc_layers(x)

        return x

In [67]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = CNN().to(device)

print("Device:", device)

Device: mps


In [68]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [72]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    model.train()
    for img, label in train_loader:
        img = img.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        
        output = model(img) # FP
        loss = criterion(output, label) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()


    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for img, label in test_loader:
            img = img.to(device)
            label = label.to(device)
            output = model(img)
            loss = criterion(output, label)

            epoch_val_loss += loss.item()

    train_loss = epoch_training_loss / len(train_loader)

    val_loss = epoch_val_loss / len(test_loader)

    print(f"epoch={epoch+1}/{epochs} & loss={train_loss}   val_loss {val_loss}")
    

epoch=1/10 & loss=0.6586889758192259   val_loss 0.6041427619498352
epoch=2/10 & loss=0.5856888926234739   val_loss 0.5385272942226509
epoch=3/10 & loss=0.5362913218037836   val_loss 0.5072813162515903
epoch=4/10 & loss=0.48280673910831584   val_loss 0.5014816558566587
epoch=5/10 & loss=0.4163173958659172   val_loss 0.487199164156256
epoch=6/10 & loss=0.349595221209115   val_loss 0.5395024629502461
epoch=7/10 & loss=0.268566750246903   val_loss 0.5759673768590237
epoch=8/10 & loss=0.17894499325032892   val_loss 0.6995596519575037
epoch=9/10 & loss=0.1145734171672114   val_loss 0.8754414903192684
epoch=10/10 & loss=0.0509886845047104   val_loss 1.0255767451278095


In [74]:
correct = 0
total = 0

with torch.no_grad():
    for img, label in test_loader:
        img = img.to(device)
        label = label.to(device)
        output = model(img)
        loss = criterion(output, label)
        _, prediction = torch.max(output, 1)

        correct += (prediction == label).sum().item()
        total += label.size(0)

    print(f"accuracy = {correct / total * 100}")

accuracy = 74.37994003815753
